# RainGuard — Module 8: Classification Improvement & Optimisation

**Objective:** improve balanced performance across the five rainfall-intensity classes used by Module 6.

This notebook:
- keeps Modules 1–7 unchanged;
- uses the same Module 4 features and chronological train/test cutoff as Module 6;
- creates a chronological validation segment from the *training period only* for model selection;
- compares class-weighting schemes and a small, controlled hyperparameter search;
- selects using **validation Macro F1 first**, then balanced accuracy;
- evaluates the selected model once on the untouched chronological test period;
- compares it with the original Module 6 saved model on the same test rows;
- saves the optimised model separately.

**Important:** Module 6 previously selected its winner using the test set. This notebook does not tune on the test set. Because the dataset is short and rare classes have few examples, validation and test metrics may be unstable. This is rainfall-intensity classification, not flood prediction.

In [ ]:
import os, json, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd) == "notebooks" else cwd
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "rainfall_features.csv")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
FIG_DIR = os.path.join(PROJECT_ROOT, "docs", "figures")
REPORT_DIR = os.path.join(PROJECT_ROOT, "reports")
CONFIG_PATH = os.path.join(MODEL_DIR, "feature_config.json")
THRESH_PATH = os.path.join(MODEL_DIR, "intensity_thresholds.json")
ORIGINAL_MODEL_PATH = os.path.join(MODEL_DIR, "rainfall_classification_model.pkl")
ORIGINAL_META_PATH = os.path.join(MODEL_DIR, "classification_model_metadata.json")
ORIGINAL_PRED_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "classification_test_predictions.csv")

for folder in [MODEL_DIR, FIG_DIR, REPORT_DIR]:
    os.makedirs(folder, exist_ok=True)

for path in [DATA_PATH, CONFIG_PATH, THRESH_PATH, ORIGINAL_MODEL_PATH, ORIGINAL_META_PATH, ORIGINAL_PRED_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file not found: {path}\nRun Modules 1, 4 and 6 successfully first.")

with open(CONFIG_PATH, encoding="utf-8") as f:
    config = json.load(f)
with open(THRESH_PATH, encoding="utf-8") as f:
    thresholds = json.load(f)
with open(ORIGINAL_META_PATH, encoding="utf-8") as f:
    original_meta = json.load(f)

ORIG_LABELS = thresholds["labels"]
ORIG_BINS = [-np.inf if i == 0 else (np.inf if b is None else b)
             for i, b in enumerate(thresholds["bins"])]

print("Project root:", PROJECT_ROOT)
print("Original Module 6 model:", original_meta["best_model"], "| weighting:", original_meta["weighting"])
print("Module 6 class labels:", original_meta["class_labels"])

## 1. Load the Module 4 feature table and reproduce the Module 6 class definition

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date", "target_date"])
MERGE_RARE = original_meta.get("merge_rule", "none") != "none"

if MERGE_RARE:
    CLASS_LABELS = ORIG_LABELS[:4] + ["Very Heavy or above"]
    CODE_MAP = {i: min(i, 4) for i in range(len(ORIG_LABELS))}
else:
    CLASS_LABELS = list(ORIG_LABELS)
    CODE_MAP = {i: i for i in range(len(ORIG_LABELS))}

N_CLASSES = len(CLASS_LABELS)
HEAVY_CODE = CLASS_LABELS.index("Heavy")
TARGET = "target_class"

df[TARGET] = df["target_next_code"].map(CODE_MAP).astype(int)
assert (df["target_date"] - df["date"]).dt.days.eq(1).all(), "Target dates are not next-calendar-day dates."
assert not df.duplicated(["state", "district", "date"]).any(), "Duplicate district/date rows found."

G = config["feature_groups"]
FEATURES = G["history"] + G["season"] + G["location_numeric"] + G["climatology"]
FORBIDDEN = ["category", "weekly", "monthly", "cumulative", "departure", "intensity", "target", "class"]
bad_features = [c for c in FEATURES if any(word in c.lower() for word in FORBIDDEN)]
if bad_features:
    raise ValueError(f"Forbidden/leakage-prone features found: {bad_features}")
missing_features = [c for c in FEATURES if c not in df.columns]
if missing_features:
    raise KeyError(f"Module 4 feature columns missing: {missing_features}")
if df[FEATURES].isna().any().any():
    print("Note: missing feature values exist; imputers inside each pipeline will handle them.")

# Reproduce Module 6's chronological cutoff exactly.
if "regression_split_date" in config:
    split_date = pd.Timestamp(config["regression_split_date"])
else:
    unique_dates = np.array(sorted(df["target_date"].unique()))
    split_date = pd.Timestamp(unique_dates[min(max(int(len(unique_dates) * 0.8), 1), len(unique_dates)-1)])

development_df = df[df["target_date"] < split_date].copy()
test_df = df[df["target_date"] >= split_date].copy()
if development_df.empty or test_df.empty:
    raise ValueError("Chronological train/test split produced an empty partition.")

print("Feature count:", len(FEATURES), FEATURES)
print("Module 6 cutoff:", split_date.date())
print("Development rows:", len(development_df), "| Test rows:", len(test_df))
display(pd.DataFrame({
    "Class": CLASS_LABELS,
    "Development rows": np.bincount(development_df[TARGET], minlength=N_CLASSES),
    "Test rows": np.bincount(test_df[TARGET], minlength=N_CLASSES)
}))

## 2. Create a chronological validation split inside the development period

The validation period is the latest 20% of unique development target dates. All districts from the same target date stay together. The final test period remains untouched until model selection is complete.

In [ ]:
dev_dates = np.array(sorted(development_df["target_date"].unique()))
if len(dev_dates) < 10:
    raise ValueError(f"Only {len(dev_dates)} development dates are available; too few for a reliable chronological validation split.")

val_start_idx = max(1, int(np.floor(len(dev_dates) * 0.80)))
val_start_idx = min(val_start_idx, len(dev_dates) - 1)
validation_start = pd.Timestamp(dev_dates[val_start_idx])

train_df = development_df[development_df["target_date"] < validation_start].copy()
val_df = development_df[development_df["target_date"] >= validation_start].copy()

X_train, y_train = train_df[FEATURES], train_df[TARGET].to_numpy()
X_val, y_val = val_df[FEATURES], val_df[TARGET].to_numpy()
X_test, y_test = test_df[FEATURES], test_df[TARGET].to_numpy()

print("Fit period:", train_df["target_date"].min().date(), "to", train_df["target_date"].max().date(), "| rows:", len(train_df))
print("Validation:", val_df["target_date"].min().date(), "to", val_df["target_date"].max().date(), "| rows:", len(val_df))
print("Final test:", test_df["target_date"].min().date(), "to", test_df["target_date"].max().date(), "| rows:", len(test_df))

class_counts = pd.DataFrame({
    "Class": CLASS_LABELS,
    "Fit rows": np.bincount(y_train, minlength=N_CLASSES),
    "Validation rows": np.bincount(y_val, minlength=N_CLASSES),
    "Final test rows": np.bincount(y_test, minlength=N_CLASSES)
})
display(class_counts)
print("Validation classes present:", [CLASS_LABELS[i] for i in np.unique(y_val)])
print("Test classes present:", [CLASS_LABELS[i] for i in np.unique(y_test)])
if len(np.unique(y_val)) < N_CLASSES:
    print("WARNING: Some classes are absent from validation. Macro F1 will still use all defined labels and report zero for absent classes.")

## 3. Define balanced evaluation metrics and the baseline

In [ ]:
ALL_CODES = list(range(N_CLASSES))

def evaluate_multiclass(y_true, y_pred, prefix=""):
    return {
        prefix + "Accuracy": accuracy_score(y_true, y_pred),
        prefix + "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        prefix + "Macro F1": f1_score(y_true, y_pred, labels=ALL_CODES, average="macro", zero_division=0),
        prefix + "Weighted F1": f1_score(y_true, y_pred, labels=ALL_CODES, average="weighted", zero_division=0),
        prefix + "Macro precision": precision_score(y_true, y_pred, labels=ALL_CODES, average="macro", zero_division=0),
        prefix + "Macro recall": recall_score(y_true, y_pred, labels=ALL_CODES, average="macro", zero_division=0)
    }

# Simple majority baseline on validation only.
majority_code = int(pd.Series(y_train).value_counts().idxmax())
majority_val_pred = np.full(len(y_val), majority_code, dtype=int)
baseline_validation = evaluate_multiclass(y_val, majority_val_pred)
print("Majority class learned from fit period:", CLASS_LABELS[majority_code])
display(pd.DataFrame([baseline_validation], index=["Majority baseline"]).round(4))

## 4. Train candidate models and weighting strategies on the fit period

We compare a small, controlled search. Weighting is applied using sample weights so the same three schemes can be compared across all estimators. No final-test data is used in fitting or selection.

In [ ]:
def make_weights(y, scheme):
    if scheme == "none":
        return np.ones(len(y), dtype=float)
    balanced = compute_sample_weight(class_weight="balanced", y=y)
    if scheme == "balanced":
        return balanced
    if scheme == "softened":
        return np.sqrt(balanced)
    raise ValueError("Unknown weighting scheme: " + scheme)

def make_candidate_models():
    imputer = lambda: SimpleImputer(strategy="median")
    candidates = []

    # Logistic Regression: tune regularisation strength.
    for C_value in [0.1, 1.0, 5.0]:
        candidates.append((
            f"Logistic Regression (C={C_value})", "Logistic Regression", C_value,
            Pipeline([("imputer", imputer()), ("scaler", StandardScaler()),
                      ("model", LogisticRegression(C=C_value, max_iter=4000, random_state=RANDOM_STATE))])
        ))

    # Decision Tree: constrain depth and leaf size to reduce overfitting.
    for depth, leaf in [(4, 30), (6, 20), (8, 30)]:
        candidates.append((
            f"Decision Tree (depth={depth}, leaf={leaf})", "Decision Tree", (depth, leaf),
            Pipeline([("imputer", imputer()),
                      ("model", DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=RANDOM_STATE))])
        ))

    # Random Forest: moderate search to control runtime.
    for depth, leaf in [(None, 5), (12, 5), (8, 10)]:
        candidates.append((
            f"Random Forest (depth={depth}, leaf={leaf})", "Random Forest", (depth, leaf),
            Pipeline([("imputer", imputer()),
                      ("model", RandomForestClassifier(n_estimators=250, max_depth=depth, min_samples_leaf=leaf,
                                                       max_features="sqrt", n_jobs=-1, random_state=RANDOM_STATE))])
        ))

    # Gradient Boosting: a small search.
    for estimators, learning_rate, depth in [(80, 0.05, 2), (120, 0.05, 2), (100, 0.03, 3)]:
        candidates.append((
            f"Gradient Boosting (n={estimators}, lr={learning_rate}, depth={depth})",
            "Gradient Boosting", (estimators, learning_rate, depth),
            Pipeline([("imputer", imputer()),
                      ("model", GradientBoostingClassifier(n_estimators=estimators, learning_rate=learning_rate,
                                                           max_depth=depth, subsample=0.8, random_state=RANDOM_STATE))])
        ))
    return candidates

WEIGHTING_SCHEMES = ["none", "softened", "balanced"]
candidate_specs = make_candidate_models()
candidate_rows = []
validation_models = {}

for model_label, family, params, pipe in candidate_specs:
    for scheme in WEIGHTING_SCHEMES:
        weights = make_weights(y_train, scheme)
        pipe.fit(X_train, y_train, model__sample_weight=weights)
        val_pred = pipe.predict(X_val)
        metrics = evaluate_multiclass(y_val, val_pred)
        row = {
            "Model": family, "Candidate": model_label, "Parameters": str(params),
            "Weighting": scheme, **metrics
        }
        candidate_rows.append(row)
        validation_models[(model_label, scheme)] = pipe
        print("Finished:", model_label, "| weighting:", scheme)

validation_results = pd.DataFrame(candidate_rows)
validation_results = validation_results.sort_values(
    ["Macro F1", "Balanced accuracy", "Weighted F1"], ascending=False
).reset_index(drop=True)
display(validation_results.round(4))

## 5. Select the best candidate using validation data only

In [ ]:
# Balanced-priority selection: Macro F1 first, then balanced accuracy, then weighted F1.
best_validation_row = validation_results.iloc[0]
best_key = (best_validation_row["Candidate"], best_validation_row["Weighting"])
best_validation_model = validation_models[best_key]

print("Selected on validation only:")
print("Candidate:", best_validation_row["Candidate"])
print("Weighting:", best_validation_row["Weighting"])
print("Validation Macro F1:", round(best_validation_row["Macro F1"], 4))
print("Validation balanced accuracy:", round(best_validation_row["Balanced accuracy"], 4))
print("Validation weighted F1:", round(best_validation_row["Weighted F1"], 4))

# Plot the leading candidates for transparent comparison.
top_plot = validation_results.head(12).copy()
top_plot["Display"] = top_plot["Model"] + " | " + top_plot["Weighting"] + " | " + top_plot["Parameters"]
fig, ax = plt.subplots(figsize=(11, 7))
sns.barplot(data=top_plot.sort_values("Macro F1"), x="Macro F1", y="Display", ax=ax)
ax.set_title("Top validation candidates – Macro F1")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m8_01_validation_candidate_comparison.png"), dpi=160, bbox_inches="tight")
plt.show()

## 6. Refit the selected configuration on the full development period

After choosing hyperparameters and weighting using validation, refit that exact configuration on all pre-test development rows. Then evaluate once on the final chronological test period.

In [ ]:
selected_spec = next(spec for spec in candidate_specs if spec[0] == best_validation_row["Candidate"])
selected_label, selected_family, selected_params, final_model = selected_spec

final_weights = make_weights(development_df[TARGET].to_numpy(), best_validation_row["Weighting"])
final_model.fit(development_df[FEATURES], development_df[TARGET].to_numpy(),
                model__sample_weight=final_weights)

optimized_test_pred = final_model.predict(X_test)
optimized_test_metrics = evaluate_multiclass(y_test, optimized_test_pred, prefix="Test ")
display(pd.DataFrame([optimized_test_metrics], index=["Optimised model"]).round(4))
print("Final test rows:", len(y_test))
print("Final test class counts:", dict(zip(CLASS_LABELS, np.bincount(y_test, minlength=N_CLASSES))))

## 7. Compare against the original Module 6 model on the same final test rows

In [ ]:
original_model = joblib.load(ORIGINAL_MODEL_PATH)
original_test_pred = original_model.predict(X_test)

original_test_metrics = evaluate_multiclass(y_test, original_test_pred, prefix="Test ")
comparison = pd.DataFrame([
    {"Model version": "Original Module 6", **original_test_metrics},
    {"Model version": "Module 8 optimised", **optimized_test_metrics}
])
display(comparison.round(4))

# Class-level recall comparison makes minority-class behaviour visible.
original_report = classification_report(y_test, original_test_pred, labels=ALL_CODES,
                                        target_names=CLASS_LABELS, output_dict=True, zero_division=0)
optimized_report = classification_report(y_test, optimized_test_pred, labels=ALL_CODES,
                                         target_names=CLASS_LABELS, output_dict=True, zero_division=0)
class_recall_comparison = pd.DataFrame({
    "Class": CLASS_LABELS,
    "Original recall": [original_report[c]["recall"] for c in CLASS_LABELS],
    "Optimised recall": [optimized_report[c]["recall"] for c in CLASS_LABELS],
    "Test support": [int(original_report[c]["support"]) for c in CLASS_LABELS]
})
display(class_recall_comparison.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
class_recall_comparison.set_index("Class")[["Original recall", "Optimised recall"]].plot(kind="bar", ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("Recall")
ax.set_title("Per-class recall: original vs optimised model")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m8_02_per_class_recall_comparison.png"), dpi=160, bbox_inches="tight")
plt.show()

## 8. Confusion matrices and detailed report

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, pred, title in [
    (axes[0], original_test_pred, "Original Module 6"),
    (axes[1], optimized_test_pred, "Module 8 optimised")
]:
    cm = confusion_matrix(y_test, pred, labels=ALL_CODES)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_LABELS,
                yticklabels=CLASS_LABELS, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m8_03_confusion_matrices.png"), dpi=160, bbox_inches="tight")
plt.show()

print("Optimised model classification report:")
print(classification_report(y_test, optimized_test_pred, labels=ALL_CODES,
                            target_names=CLASS_LABELS, zero_division=0, digits=4))

## 9. Heavy-or-above diagnostic (secondary metric, not the selection objective)

In [ ]:
# This is a binary view derived from the five-class predictions.
# It is reported for context; model selection remains balanced across all classes.
def heavy_view(y_true, y_pred):
    yt = (np.asarray(y_true) >= HEAVY_CODE).astype(int)
    yp = (np.asarray(y_pred) >= HEAVY_CODE).astype(int)
    return {
        "Heavy+ precision": precision_score(yt, yp, zero_division=0),
        "Heavy+ recall": recall_score(yt, yp, zero_division=0),
        "Heavy+ F1": f1_score(yt, yp, zero_division=0),
        "Heavy+ actual": int(yt.sum()),
        "Heavy+ detected": int(((yt == 1) & (yp == 1)).sum()),
        "Heavy+ false alarms": int(((yt == 0) & (yp == 1)).sum())
    }

heavy_comparison = pd.DataFrame([
    {"Model version": "Original Module 6", **heavy_view(y_test, original_test_pred)},
    {"Model version": "Module 8 optimised", **heavy_view(y_test, optimized_test_pred)}
])
display(heavy_comparison.round(4))
print("Heavy+ metrics are secondary diagnostics. The primary selection rule is Macro F1, then balanced accuracy.")

## 10. Save the optimised model and complete evaluation artifacts

In [ ]:
OPT_MODEL_PATH = os.path.join(MODEL_DIR, "rainfall_classification_model_optimized.joblib")
OPT_META_PATH = os.path.join(MODEL_DIR, "classification_optimization_metadata.json")
VAL_RESULTS_PATH = os.path.join(REPORT_DIR, "rainguard_module8_validation_results.csv")
TEST_COMPARE_PATH = os.path.join(REPORT_DIR, "rainguard_module8_test_comparison.csv")
CLASS_RECALL_PATH = os.path.join(REPORT_DIR, "rainguard_module8_class_recall.csv")
HEAVY_COMPARE_PATH = os.path.join(REPORT_DIR, "rainguard_module8_heavy_diagnostic.csv")
PRED_OUT_PATH = os.path.join(DATA_DIR, "classification_optimized_test_predictions.csv")

joblib.dump(final_model, OPT_MODEL_PATH)
validation_results.to_csv(VAL_RESULTS_PATH, index=False)
comparison.to_csv(TEST_COMPARE_PATH, index=False)
class_recall_comparison.to_csv(CLASS_RECALL_PATH, index=False)
heavy_comparison.to_csv(HEAVY_COMPARE_PATH, index=False)

pred_out = test_df[["state", "district", "date", "target_date"]].copy()
pred_out["actual_code"] = y_test
pred_out["actual_class"] = [CLASS_LABELS[i] for i in y_test]
pred_out["original_pred_code"] = original_test_pred
pred_out["original_pred_class"] = [CLASS_LABELS[i] for i in original_test_pred]
pred_out["optimized_pred_code"] = optimized_test_pred
pred_out["optimized_pred_class"] = [CLASS_LABELS[i] for i in optimized_test_pred]
pred_out.to_csv(PRED_OUT_PATH, index=False)

metadata = {
    "project": "RainGuard",
    "module": 8,
    "task": "balanced next-calendar-day rainfall intensity classification",
    "class_labels": CLASS_LABELS,
    "features": FEATURES,
    "chronological_test_cutoff": str(split_date.date()),
    "validation_start": str(validation_start.date()),
    "selected_candidate": str(selected_label),
    "selected_family": str(selected_family),
    "selected_parameters": str(selected_params),
    "weighting": str(best_validation_row["Weighting"]),
    "selection_rule": "highest validation Macro F1, then validation balanced accuracy, then weighted F1",
    "validation_metrics": {k: float(v) for k, v in evaluate_multiclass(y_val, best_validation_model.predict(X_val)).items()},
    "test_metrics": {k: float(v) for k, v in optimized_test_metrics.items()},
    "test_heavy_diagnostic": {k: (float(v) if isinstance(v, (np.floating, float)) else int(v))
                              for k, v in heavy_view(y_test, optimized_test_pred).items()},
    "note": "Test set was not used for hyperparameter or candidate selection in Module 8. Dataset is short; results are provisional. This is rainfall intensity classification, not flood prediction."
}
with open(OPT_META_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# Verify saved model reproduces the optimised predictions.
reloaded = joblib.load(OPT_MODEL_PATH)
assert np.array_equal(reloaded.predict(X_test), optimized_test_pred)

print("Saved optimised model:", OPT_MODEL_PATH)
print("Saved metadata:", OPT_META_PATH)
print("Saved validation results:", VAL_RESULTS_PATH)
print("Saved test comparison:", TEST_COMPARE_PATH)
print("Saved class recall comparison:", CLASS_RECALL_PATH)
print("Saved heavy diagnostic:", HEAVY_COMPARE_PATH)
print("Saved test predictions:", PRED_OUT_PATH)
print("Reload verification passed.")

## 11. How to decide whether Module 8 improved the model

Do **not** assume the optimised model is better just because it is new.

- Compare **Test Macro F1** and **Test balanced accuracy** against the original Module 6 model.
- Inspect each class's recall, especially the rare top class.
- Review the confusion matrices and heavy-rain false alarms.
- If the optimised model does not improve balanced metrics, retain the original model and document the result honestly.
- The final test set is small and has few heavy-rain observations, so a small score difference may not be meaningful.

### Completion checklist
- [ ] Validation split is chronological and entirely before the final test period.
- [ ] Candidate selection uses validation Macro F1 and balanced accuracy.
- [ ] Final test comparison is run once after selection.
- [ ] Per-class recall and confusion matrices reviewed.
- [ ] Optimised model saved separately; Module 6 artifacts remain unchanged.
- [ ] Results and limitations documented.

**Next:** use the evidence from Module 8 to decide whether to retain the original classifier or adopt the optimised one. Then proceed to final interpretation and project documentation.